<a href="https://colab.research.google.com/github/GlobalFishingWatch/gfw-api-python-client/blob/develop/notebooks/workflow-guides/workflow-01-analyze-apparent-fishing-effort-senegalese-eez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analyze apparent fishing effort in Senegalese EEZ

This guide provides detailed instructions to on how to use the [gfw-api-python-client](https://github.com/GlobalFishingWatch/gfw-api-python-client) to **Analyze apparent fishing effort in [Senegalese EEZ](https://www.marineregions.org/gazetteer.php?p=details&id=8371) region and monitor vessel activities** using **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)**, and **[Vessels API](https://globalfishingwatch.org/our-apis/documentation#vessels-api)**.

**Note:** See the [Datasets](https://globalfishingwatch.org/our-apis/documentation#api-dataset), [Data Caveats](https://globalfishingwatch.org/our-apis/documentation#data-caveat), and [Terms of Use](https://globalfishingwatch.org/our-apis/documentation#terms-of-use) pages in the [GFW API documentation](https://globalfishingwatch.org/our-apis/documentation#introduction) for details on GFW data, API licenses, and rate limits.

## Prerequisites

Before using the `gfw-api-python-client`, ensure it is installed (see the [Getting Started](https://globalfishingwatch.github.io/gfw-api-python-client/getting-started.html) guide) and that you have obtained an API access token from the [Global Fishing Watch API portal](https://globalfishingwatch.org/our-apis/tokens).

## Installation

The `gfw-api-python-client` can be installed easily from either the [Python Package Index (PyPI)](https://pypi.org/project/gfw-api-python-client/) or [Conda](https://anaconda.org/conda-forge/gfw-api-python-client)

In [1]:
# %pip install gfw-api-python-client

In [1]:
# %conda install -c conda-forge gfw-api-python-client

## Usage

Import and use `gfw-api-python-client` in your Python codes

In [2]:
import datetime
import os

import pandas as pd

import gfwapiclient as gfw

In [3]:
try:
    from google.colab import userdata

    access_token = userdata.get("GFW_API_ACCESS_TOKEN")
except Exception:
    access_token = os.environ.get("GFW_API_ACCESS_TOKEN")

access_token = access_token or "<PASTE_YOUR_GFW_API_ACCESS_TOKEN_HERE>"

In [4]:
gfw_client = gfw.Client(
    access_token=access_token,
)

## Introduction

**Use Case: A Port Inspector Monitoring Vessel Activity**

Mamadou, a port inspector in Dakar, Senegal, monitors vessel activity within **[Senegalese Exclusive Economic Zone (EEZ)]((https://www.marineregions.org/gazetteer.php?p=details&id=8371))**. His goal is to:

1. Analyzing apparent fishing effort, specifically for **trawlers** in Senegalese EEZ.
2. Identifying vessels involved in **apparent trawling activity** and determining their reported **flag states**.
3. Checking vessel history, including prior **encounters (or potential transshipment)** or **port visits**.
4. Generating reports for enforcement authorities to assess risks.

**APIs Used:**
️
1. **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)** – To retrieve **apparent fishing effort** data for trawlers operating in Senegalese EEZ over the past 3 months.
2. **[Vessels API](https://globalfishingwatch.org/our-apis/documentation#vessels-api)** – To retrieve **detailed vessel information**, including `flag`, `ownership history`, and `authorizations`.

**Important:** In order to avoid any misinterpretation of **GFW data**, please refer to our official **data caveats** documentations:
- [Apparent fishing effort](https://globalfishingwatch.org/dataset-and-code-fishing-effort/) 
- [Exclusive economic zone boundaries definition](https://globalfishingwatch.org/our-apis/documentation#exclusive-economic-zone-boundaries-definition)
- [Vessel ID](https://globalfishingwatch.org/our-apis/documentation#vessel-id)
- [Vessel API - Vessel identity information](https://globalfishingwatch.org/our-apis/documentation#vessel-api-vessel-identity-information)

**Important Caveats:**

1. The [4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api) only supports **one active report per user at a time**.
2. **Sending multiple requests simultaneously** results in a **429 Too Many Requests** error.
3. If a report takes over **100 seconds** to generate, it may return a **524 Gateway Timeout** error.

## Step 0: Identify the Region of Interest (ROI) - Senegalese EEZ

Before making API requests, Mamadou must specify the geographic area for analysis using a **Region ID**:

**Options to Define the Region:**

1. **Using Region ID** - Each EEZ has a unique ID in the **[public-eez-areas](https://globalfishingwatch.org/our-apis/documentation#regions)** dataset.
2. **Custom Geometries** - Users can define a custom area using GeoJSON.
   
For **[Senegalese EEZ, the region ID is 8371](https://www.marineregions.org/gazetteer.php?p=details&id=8371)** (public-eez-areas dataset).

**Note:** See how to use the [Reference Data API - Usage Guides](https://globalfishingwatch.github.io/gfw-api-python-client/usage-guides/references-data-api.html) to obtain and filter predefined [**Regions of Interest (ROIs)**](https://globalfishingwatch.org/our-apis/documentation#regions), such as Exclusive Economic Zones (**EEZs**), Marine Protected Areas (**MPAs**), and Regional Fisheries Management Organizations (**RFMOs**).

In [5]:
eez_rois_result = await gfw_client.references.get_eez_regions(iso3="SEN")
sen_eez_roi = eez_rois_result.data()[0]

In [6]:
sen_eez_roi.id, sen_eez_roi.dataset, sen_eez_roi.label, sen_eez_roi.iso3

('8371', 'public-eez-areas', 'Senegalese Exclusive Economic Zone', 'SEN')

## Step 1: Retrieve Apparent Fishing Effort in Senegalese EEZ

Mamadou **first queries** the **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)** to get **apparent fishing effort for all vessels**, grouping them by **vessel ID** in **[Senegalese EEZ](https://www.marineregions.org/gazetteer.php?p=details&id=8371)**. Please [learn more about apparent fishing effort here](https://globalfishingwatch.org/our-apis/documentation#ais-apparent-fishing-effort) and [check its data caveats here](https://globalfishingwatch.org/our-apis/documentation#apparent-fishing-effort).

**Filters Used:**

1. **[Region ID](https://globalfishingwatch.org/our-apis/documentation#regions)** - [8371 Senegalese EEZ]((https://www.marineregions.org/gazetteer.php?p=details&id=8371))
2. **[Date Range](https://globalfishingwatch.org/our-apis/documentation#report-url-parameters-for-both-post-and-get-requests)** - Last 3 Months
3. **[Grouped By](https://globalfishingwatch.org/our-apis/documentation#report-url-parameters-for-both-post-and-get-requests)** - Vessel ID
4. **[Gear Type](https://globalfishingwatch.org/our-apis/documentation#gear-types-supported)** - Trawlers 

**Why Use group-by=VESSEL_ID?**

Grouping by **VESSEL_ID** allows **individual vessel identification** in the response. This is crucial for **tracking vessel activity** and, more importantly, linking each detected vessel to the **[Vessels API](https://globalfishingwatch.org/our-apis/documentation#vessels-api)** in the next step. By structuring the query this way, we can fetch vessel details such as **flag, name, and ownership records** in **Step 2 below**.


**Explanation of Parameters & Considerations**

- Gear types, such as **trawlers**, are inferred based on **Global Fishing Watch’s vessel classification system**, which relies on **AIS data and vessel public registries**. The **gear type associated with each vessel is not always 100% accurate**, as it may be derived from historical sources or inferred from movement patterns. See more details on [supported gear types here](https://globalfishingwatch.org/our-apis/documentation#gear-types-supported).
- Also, please see data caveats regarding [vessel types and their classification here](https://globalfishingwatch.org/our-apis/documentation#vessel-types).
- See [Reference Data API - Usage Guides](https://globalfishingwatch.github.io/gfw-api-python-client/usage-guides/references-data-api.html) for more details on retrieving [Region IDs](https://globalfishingwatch.org/our-apis/documentation#regions).

In [7]:
end_date = datetime.date.today()

In [8]:
start_date = end_date - datetime.timedelta(weeks=12)

In [9]:
start_date, end_date

(datetime.date(2026, 4, 2), datetime.date(2026, 6, 25))

In [10]:
step_1_report_result = await gfw_client.fourwings.create_fishing_effort_report(
    spatial_resolution="HIGH",
    group_by="VESSEL_ID",
    temporal_resolution="MONTHLY",
    filters=["geartype in ('trawlers')"],
    start_date=start_date,
    end_date=end_date,
    spatial_aggregation=True,
    region=sen_eez_roi,
)

In [11]:
step_1_report_df = step_1_report_result.df()

In [12]:
step_1_report_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 211 entries, 0 to 210
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype              
---  ------                   --------------  -----              
 0   date                     211 non-null    str                
 1   detections               0 non-null      object             
 2   flag                     211 non-null    str                
 3   gear_type                211 non-null    str                
 4   hours                    211 non-null    float64            
 5   vessel_ids               0 non-null      object             
 6   vessel_id                211 non-null    str                
 7   vessel_type              211 non-null    str                
 8   entry_timestamp          211 non-null    datetime64[us, UTC]
 9   exit_timestamp           211 non-null    datetime64[us, UTC]
 10  first_transmission_date  211 non-null    datetime64[us, UTC]
 11  last_transmission_date   211 non-null    da

In [13]:
step_1_report_df[["flag", "gear_type", "hours", "mmsi", "ship_name"]].head()

,flag,gear_type,hours,mmsi,ship_name
0,SEN,TRAWLERS,528.425000,663152000,KENTIA
1,GNB,TRAWLERS,135.772500,630124008,BACALAM
2,SEN,TRAWLERS,330.992222,663092000,SOKONE
3,CHN,TRAWLERS,543.411111,412549197,DAK 1372
4,SEN,TRAWLERS,324.623889,663146000,F/V AUDREY-


### Explore Vessels Potentially Engaged in Trawling Activity in the Senegalese EEZ

In [14]:
step_1_agg_report_df = (
    step_1_report_df.groupby(["flag", "gear_type", "mmsi", "ship_name"], as_index=False)
    .agg(hours=("hours", "sum"))
    .sort_values(by="hours", ascending=False)
)

In [15]:
step_1_agg_report_df["hours"].describe()

count      88.000000
mean      567.947727
std       612.831612
min         0.318611
25%        58.696042
50%       220.915694
75%      1254.456458
max      1680.045556
Name: hours, dtype: float64

In [16]:
step_1_agg_report_mask = step_1_agg_report_df["hours"] >= step_1_agg_report_df[
    "hours"
].quantile(0.75)

In [17]:
step_1_agg_report_df[step_1_agg_report_mask]

,flag,gear_type,mmsi,ship_name,hours
54,SEN,TRAWLERS,663093000,AMINE,1680.045556
59,SEN,TRAWLERS,663111111,LAGUEM I,1650.861667
55,SEN,TRAWLERS,663101000,CHIQUITA,1629.070833
57,SEN,TRAWLERS,663103000,RIA DE DAKAR,1614.329722
77,SEN,TRAWLERS,663176000,CARVISA DOS,1596.626389
69,SEN,TRAWLERS,663131000,KANBAL II,1588.290556
48,SEN,TRAWLERS,663010400,TOUBA,1583.635278
80,SEN,TRAWLERS,663180000,F/V NATA,1582.401944
70,SEN,TRAWLERS,663133000,"KANBAL III ""3",1582.378056
85,SEN,TRAWLERS,663250000,PRAIA DA MAROSA,1561.656944


### What We have Learned from Step 1

- There are vessels appear to have been engaged in potential **trawling activity** in Senegalese EEZ over the past 3 months.
- We will retrieve these vessels' `ownership`, `flag history`, and `authorizations` in **Step 2 to validate** them.

## Step 2: Retrieve Vessel Details Using the Vessels API

Mamadou queries the **[Vessels API](https://globalfishingwatch.org/our-apis/documentation#vessels-api)** to **get detailed vessel identity and ownership records**. Please [learn more about Vessels API here](https://globalfishingwatch.org/our-apis/documentation#vessels-api) and [check its data caveats here](https://globalfishingwatch.org/our-apis/documentation#vessel-api-vessel-identity-information).

**Filters Used:**

1. **Vessel IDs** from [4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api), **Step 1 above**.
2. **[Datasets](https://globalfishingwatch.org/our-apis/documentation#api-dataset)** - `public-global-vessel-identity:latest`.
3. **[Includes](https://globalfishingwatch.org/our-apis/documentation#get-vessels-by-ids-url-parameters)** - `POTENTIAL_RELATED_SELF_REPORTED_INFO`.

**Note:** Vessels may change identifiers over time, such as their `Maritime Mobile Service Identity (MMSI)`,` International Maritime Organization (IMO) number)`, `call sign`, or even their `name`. These changes can occur due to `re-registration`, `changes in ownership`, or other `operational reasons` within the `AIS transponder`. Parameter (`includes = POTENTIAL_RELATED_SELF_REPORTED_INFO`) helps group all **vessel ids** that are **potentially related** as part of the **same physical vessel** based on publicly available registry information.

In [18]:
step_1_vessel_mmsis = list(step_1_agg_report_df[step_1_agg_report_mask]["mmsi"])

In [19]:
step_1_vessel_mmsis

['663093000',
 '663111111',
 '663101000',
 '663103000',
 '663176000',
 '663131000',
 '663010400',
 '663180000',
 '663133000',
 '663250000',
 '412549196',
 '412549197',
 '663178000',
 '412420883',
 '663114000',
 '663113000',
 '663073000',
 '663112000',
 '663109000',
 '663115000',
 '663122000',
 '663152000']

In [20]:
step_1_vessel_ids = list(
    step_1_report_df[step_1_report_df["mmsi"].isin(step_1_vessel_mmsis)][
        "vessel_id"
    ].unique()
)

In [21]:
step_1_vessel_ids

['b289909a3-3fe7-d20b-ff71-e0faae7cfbd1',
 '50009f324-4ad6-bd1f-55ce-91adcbe14835',
 '3d4bfcfd2-23c1-3379-d4c2-63239c90b40e',
 'f5d810a8d-d406-64e0-fc57-8f2edbc0894c',
 'bf28c5a58-8c83-8690-8689-7f2d520f926e',
 '26aa49b9d-d1eb-aa18-71a8-49cc406c4d0f',
 '7bfa4e72a-aa54-4aae-0aa6-145fc83a25bb',
 '56797171d-dc16-997d-5765-61029b1e0244',
 '90ab31dfb-bcab-a05f-d12f-2544e1869205',
 '84c3d602c-cc35-8137-7c15-e3ff55b8b3c7',
 'bd0fd660c-ce55-f36f-ebf5-5bd0126c1d3e',
 '833e0dfb5-52fe-cf51-e0e5-c52f024ebdac',
 'e1978b237-794c-1a7d-171c-6c11f629c154',
 '54423a274-4798-9a62-79f7-4c80605246ac',
 '894bc3ec6-6ade-f09c-e792-ff2e947508d8',
 '32e3e6c0a-aa3e-b7f8-7ac4-0b557fc601a8',
 'cee15c9d1-1057-e347-f773-8a64a32bc08a',
 '0caeaa7c7-71e8-b911-3acb-8f01fc9eefe6',
 '2dcdb9a93-3782-9140-78eb-9d55e8b4e3d3',
 '8a228af50-03a0-6abc-8dd5-08699a638bcb',
 'd0e58b8b6-6c5d-c117-d258-c0a38bfdcff4',
 'c6ca30d4c-cca8-92c9-b821-6620ee18940b']

In [22]:
step_2_vessels_result = await gfw_client.vessels.get_vessels_by_ids(
    ids=step_1_vessel_ids,
)

In [23]:
step_2_vessels_df = step_2_vessels_result.df()

In [24]:
step_2_vessels_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 23 entries, 0 to 22
Data columns (total 7 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   dataset                         23 non-null     str   
 1   registry_info_total_records     23 non-null     int64 
 2   registry_info                   23 non-null     object
 3   registry_owners                 23 non-null     object
 4   registry_public_authorizations  23 non-null     object
 5   combined_sources_info           23 non-null     object
 6   self_reported_info              23 non-null     object
dtypes: int64(1), object(5), str(1)
memory usage: 1.4+ KB


**Understanding Vessel Details Response Data**

- **registryInfoTotalRecords** – This represents the **number of registry records** found for the vessels.
- **registryInfo** – Contains **public registry data**. This data is sourced from official **vessel registries**.
- **registryOwners** – Lists the **registered owners** of the vessel based on public sources.
- **registryPublicAuthorizations** – Represents known **fishing authorizations** from public sources. Users should verify against national registries and RFMO records for additional context.
- **combinedSourcesInfo** – Provides inferred data from multiple sources, including. This is not explicitly reported by vessels but determined through **GFW's classification methods**.
- **selfReportedInfo** – Contains **AIS self-reported** data, including `MMSI`, `ship name`, and `flag` as broadcast by the **vessel itself**. Self-reported data may not always align with registry data and should be cross-checked.

In [25]:
step_2_vessels_df[["registry_info", "registry_owners", "self_reported_info"]]

,registry_info,registry_owners,self_reported_info
0,[],[],[{'id': 'b289909a3-3fe7-d20b-ff71-e0faae7cfbd1...
1,[],[],[{'id': 'bf28c5a58-8c83-8690-8689-7f2d520f926e...
2,"[{'id': '4040e99d8c9daabbc124ad166611b46c', 's...","[{'name': 'SOPERKA', 'flag': 'SEN', 'ssvid': '...",[{'id': '8a228af50-03a0-6abc-8dd5-08699a638bcb...
3,[],[],[{'id': 'e1978b237-794c-1a7d-171c-6c11f629c154...
4,[],[],[{'id': '32e3e6c0a-aa3e-b7f8-7ac4-0b557fc601a8...
5,[],[],[{'id': 'bd0fd660c-ce55-f36f-ebf5-5bd0126c1d3e...
6,[],[],[{'id': '56797171d-dc16-997d-5765-61029b1e0244...
7,[],[],[{'id': '56797171d-dc16-997d-5765-61029b1e0244...
8,[],[],[{'id': '2dcdb9a93-3782-9140-78eb-9d55e8b4e3d3...
9,[],[],[{'id': '833e0dfb5-52fe-cf51-e0e5-c52f024ebdac...


### Explore Vessels Registry Info

In [26]:
step_2_has_registry_info_mask = step_2_vessels_df[
    "registry_info"
].notna() & step_2_vessels_df["registry_info"].astype(bool)

In [27]:
step_2_registry_info_df = pd.json_normalize(
    step_2_vessels_df[step_2_has_registry_info_mask]["registry_info"].explode()
)

In [28]:
step_2_registry_info_df[
    ["ssvid", "flag", "ship_name", "n_ship_name", "gear_types", "source_code"]
]

,ssvid,flag,ship_name,n_ship_name,gear_types,source_code
2,663131000,SEN,KANBAL II,KANBAL2,[TRAWLERS],"[IMO, SNP]"
2,663000000,SEN,KANBAL II,KANBAL2,[TRAWLERS],"[IMO, SNP]"
20,224097970,ESP,PRAIA DA MAROSA,PRAIADAMAROSA,[TRAWLERS],"[ESP, EU, ICCAT, IMO, ISSF, SNP]"
22,663111111,SEN,LAGHEM I,LAGUEM1,[TRAWLERS],"[IMO, SNP]"
22,663111111,SEN,LAGHEM I,LAGHEM1,[TRAWLERS],"[IMO, SNP]"


### Explore Registry Owners

In [29]:
step_2_has_registry_owners_mask = step_2_vessels_df[
    "registry_owners"
].notna() & step_2_vessels_df["registry_owners"].astype(bool)

In [30]:
step_2_registry_owners_df = pd.json_normalize(
    step_2_vessels_df[step_2_has_registry_owners_mask]["registry_owners"].explode()
)

In [31]:
step_2_registry_owners_match_registry_info_mask = step_2_registry_owners_df[
    "ssvid"
].isin(step_2_registry_info_df["ssvid"])

In [32]:
step_2_registry_owners_df[step_2_registry_owners_match_registry_info_mask][
    ["ssvid", "flag", "name", "source_code"]
]

,ssvid,flag,name,source_code
2,663131000,SEN,SOPERKA,"[IMO, SNP]"
2,663000000,SEN,SOPERKA,"[IMO, SNP]"
20,224097970,ESP,ARMADORES DO MAROSA,"[ICCAT, SNP]"
22,663111111,SEN,SOPERKA,"[IMO, SNP]"


### Explore Vessels Self Reported Info

In [33]:
step_2_has_self_reported_info_mask = step_2_vessels_df[
    "self_reported_info"
].notna() & step_2_vessels_df["self_reported_info"].astype(bool)

In [34]:
step_2_self_reported_info_df = pd.json_normalize(
    step_2_vessels_df[step_2_has_self_reported_info_mask][
        "self_reported_info"
    ].explode()
)

In [35]:
step_2_self_reported_info_match_registry_info_mask = step_2_self_reported_info_df[
    "ssvid"
].isin(step_2_registry_info_df["ssvid"])

In [36]:
step_2_self_reported_info_df[step_2_self_reported_info_match_registry_info_mask][
    ["ssvid", "flag", "ship_name", "n_ship_name", "source_code"]
]

,ssvid,flag,ship_name,n_ship_name,source_code
2,663131000,SEN,KANBAL II,KANBAL2,[AIS]
2,663131000,SEN,KANBAL II,KANBAL2,[AIS]
2,663131000,SEN,KAMBAL 2,KAMBAL2,[AIS]
2,663000000,SEN,KING CRAB,KINGCRAB,[AIS]
20,224097970,ESP,PRAIA DA MAROSA,PRAIADAMAROSA,[AIS]
22,663111111,SEN,LAGUEM I,LAGUEM1,[AIS]
22,663111111,SEN,LAGHEM 1,LAGHEM1,[AIS]
22,663111111,SEN,LAGHEM 1,LAGHEM1,[AIS]
22,663111111,SEN,LAGHEM 1,LAGHEM1,[AIS]


### What We have Learned from Step 2

- **Vessel Identity:**
  - `NUEVONOSOLAR (mmsi: 663178000, flag: SEN)`- appears to be registered under Senegal (SEN)
  - `BETTY (mmsi: 663115000, flag: SEN)` - appears to be registered under Senegal (SEN)
- **Ownership & Historical Changes:**
  - `KANBAL II (mmsi: 663131000, flag: SEN)` - **SOPERKA** appears to be listed as the registered owner.
  - `LAGUEM I/LAGHEM 1 (mmsi: 663111111, flag: SEN)`- **SOPERKA** appears to be listed as the registered owner.

**Next Steps:**

- Further, **validate ownership history** using official registry sources.
- Assess whether any **historical changes** in `flag`, `name`, or `ownership` are relevant for enforcement.
- Generate an **apparent activity report** with all available details.

## Summary of API Flow

1. **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)** - Retrieve apparent fishing effort for **trawlers** within Senegalese EEZ.
2. **[Vessels API](https://globalfishingwatch.org/our-apis/documentation#vessels-api)** - Fetch detailed **vessel identity**, **ownership history**, and **public authorizations** for vessels detected in **Step 1**.
3. **Analyze vessel history** - Compare **registry records**, **AIS self-reported** data, and inferred information to identify potential flag-hopping or historical changes in vessel identity.
4. **Assess authorizations** - Cross-check whether vessels have publicly available fishing authorizations and consider external official sources for further verification.
5. **Generate an analysis report** - Provide enforcement authorities with a structured report highlighting vessel activity, identity records, and any notable discrepancies for further investigation.